In [1]:
import pandas as pd
import os
import re
import numpy as np

In [2]:
# Basically, running the steps below will complete preparation.

# Resolve the PROJECT root (not the notebook folder)
def find_project_root(start_dir):
    current = os.path.abspath(start_dir)

    while True:
        has_main = os.path.isfile(os.path.join(current, 'sources', 'main.py'))
        has_data = os.path.isdir(os.path.join(current, 'data'))

        if has_main and has_data:
            return current

        parent = os.path.dirname(current)
        if parent == current:
            return None
        current = parent


PROJECT_ROOT = find_project_root(os.getcwd())

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find project root containing both 'sources/main.py' and 'data'."
    )

BASE_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw_data')
CLEAN_DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'clean_data')

if not os.path.isdir(BASE_DIR):
    raise FileNotFoundError(f"Missing folder: {BASE_DIR}")
if not os.path.isdir(CLEAN_DATA_DIR):
    raise FileNotFoundError(f"Missing folder: {CLEAN_DATA_DIR}")

print(f'Using project root: {PROJECT_ROOT}')
print(f'Using raw data folder: {BASE_DIR}')
print(f'Using clean data folder: {CLEAN_DATA_DIR}')

Using project root: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel
Using raw data folder: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\raw_data
Using clean data folder: c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data


## OpenGo insoles
- remove 9 header rows
- remove unnecessary data
- resample from 100 Hz to 60 Hz by linear interpolation

In [3]:
# Data cleaner
# - Remove unnecessary data
# - Reshape data at 1/60-second intervals

data_path = os.path.join(BASE_DIR, 'Insoles')
output_path = CLEAN_DATA_DIR

if not os.path.isdir(data_path):
    raise FileNotFoundError(f"Missing folder: {data_path}")

csv_files = [f for f in os.listdir(data_path) if f.endswith('.txt')]

def resample_and_interpolate(data, interp_limit=6, edge_fill_limit=2):
    # Explicitly specify the timestamp column name
    timestamp_col = '# time'
    if timestamp_col not in data.columns:
        print(f"Error: Timestamp column '{timestamp_col}' does not exist in the data.")
        return None, None

    data = data.copy()
    data[timestamp_col] = pd.to_numeric(data[timestamp_col], errors='coerce')
    data = data.dropna(subset=[timestamp_col]).sort_values(timestamp_col)

    if data.empty:
        print("Error: No valid timestamp values were found.")
        return None, None

    # Force sensor columns to numeric before resampling
    sensor_cols = [c for c in data.columns if c != timestamp_col]
    data[sensor_cols] = data[sensor_cols].apply(pd.to_numeric, errors='coerce')

    na_before = int(data[sensor_cols].isna().sum().sum())

    # Use the time column as the resampling index in seconds
    data[timestamp_col] = pd.to_timedelta(data[timestamp_col], unit='s')
    data = data.set_index(timestamp_col)
    data = data[~data.index.duplicated(keep='first')]

    # Resample to 60 Hz
    data_resampled = data.resample('16.666667ms').mean()

    # Step 1: fill only short internal gaps
    data_resampled = data_resampled.interpolate(
        method='linear',
        limit=interp_limit,
        limit_direction='both',
        limit_area='inside'
    )
    na_after_interp = int(data_resampled.isna().sum().sum())

    # Step 2: cautiously fill short boundary gaps only
    if na_after_interp > 0:
        data_resampled = data_resampled.ffill(limit=edge_fill_limit).bfill(limit=edge_fill_limit)
    na_after_edge_fill = int(data_resampled.isna().sum().sum())

    # Step 3: last resort for any remaining sparse NaNs
    if na_after_edge_fill > 0:
        col_medians = data_resampled.median(numeric_only=True)
        data_resampled = data_resampled.fillna(col_medians)
    na_final = int(data_resampled.isna().sum().sum())

    # Convert the resampled index back to seconds
    data_resampled = data_resampled.reset_index()
    data_resampled[timestamp_col] = data_resampled[timestamp_col].dt.total_seconds()

    stats = {
        'na_before': na_before,
        'na_after_interp': na_after_interp,
        'na_after_edge_fill': na_after_edge_fill,
        'na_final': na_final,
    }

    return data_resampled, stats

for csv_file in csv_files:
    # Read CSV (9 rows in the header, tab-separated values)
    file_path = os.path.join(data_path, csv_file)
    data = pd.read_csv(file_path, header=9, sep='\t')

    # Normalize header formatting (some files include leading/trailing spaces)
    data.columns = data.columns.str.strip()

    # Remove unnecessary data (drop what exists, even if some columns are absent)
    cols_to_drop = [
        "left total force[N]", "left center of pressure X[-0.5...+0.5]", "left center of pressure Y[-0.5...+0.5]",
        "right total force[N]", "right center of pressure X[-0.5...+0.5]", "right center of pressure Y[-0.5...+0.5]",
        "right steps[]", "left steps[]"
    ]
    existing_cols = [c for c in cols_to_drop if c in data.columns]
    if existing_cols:
        data = data.drop(columns=existing_cols)
    else:
        print(f"Warning: No target columns found to drop in {csv_file}.")

    # Apply resampling + bounded NA handling
    data_resample, na_stats = resample_and_interpolate(data.copy())

    if data_resample is not None:
        # Save output file
        output_file_path = os.path.join(output_path, csv_file)
        data_resample.to_csv(output_file_path, index=False)

        # Confirm processing result
        print(f"Processed {csv_file}: removed {len(existing_cols)} columns")
        print(
            f"NA stats -> before: {na_stats['na_before']}, "
            f"after interp: {na_stats['na_after_interp']}, "
            f"after edge fill: {na_stats['na_after_edge_fill']}, "
            f"final: {na_stats['na_final']}"
        )
        print(data_resample.head())
    else:
        print(f'Error processing {csv_file}. Skipping.')


Processed Soles_001CcSs_3_shadow_test.txt: removed 6 columns
NA stats -> before: 242, after interp: 0, after edge fill: 0, final: 0
       # time  left pressure 1[N/cm²]  left pressure 2[N/cm²]  \
0  118.480000                     0.0                    0.00   
1  118.496667                     0.0                    0.00   
2  118.513333                     0.0                    0.00   
3  118.530000                     0.0                    0.00   
4  118.546667                     0.0                    0.25   

   left pressure 3[N/cm²]  left pressure 4[N/cm²]  left pressure 5[N/cm²]  \
0                   0.000                   0.000                   0.000   
1                   0.000                   0.000                   0.000   
2                   0.000                   0.000                   0.125   
3                   0.000                   0.000                   0.500   
4                   0.125                   0.125                   1.375   

   left pressu

Processed Soles_001CcSs_3_shadow_training.txt: removed 6 columns
NA stats -> before: 1606, after interp: 0, after edge fill: 0, final: 0
     # time  left pressure 1[N/cm²]  left pressure 2[N/cm²]  \
0  1.670000                   3.000                   1.375   
1  1.686667                   3.250                   1.250   
2  1.703333                   3.125                   1.250   
3  1.720000                   3.000                   1.250   
4  1.736667                   3.000                   1.250   

   left pressure 3[N/cm²]  left pressure 4[N/cm²]  left pressure 5[N/cm²]  \
0                   2.875                   1.625                   7.500   
1                   2.500                   1.375                   7.375   
2                   2.500                   1.250                   8.250   
3                   2.750                   1.250                   8.000   
4                   2.500                   1.500                   8.000   

   left pressure 6[N/

## Awinda IMMU
- export the joints positions tab of xlsx file into csv
- rename cols by X.#, Y.#, Z.#, ... with # a digit representing a joint

In [4]:
# Data cleaner
# - Read Segment Position tab directly from xlsx files
# - Change joint columns to X.0, Y.0, Z.0, X.1, ... , X.22, Y.22, Z.22

# Put raw data into the Awinda folder
data_path = os.path.join(BASE_DIR, 'Awinda')
output_path = CLEAN_DATA_DIR

if not os.path.isdir(data_path):
    raise FileNotFoundError(f"Missing folder: {data_path}")

# Get all xlsx files in the folder
xlsx_files = [f for f in os.listdir(data_path) if f.endswith('.xlsx')]


def rename_awinda_columns(data):
    coord_columns = [
        col for col in data.columns
        if re.search(r'\s[xyz]$', str(col), flags=re.IGNORECASE)
    ]
    expected_coord_columns = 23 * 3

    if len(coord_columns) != expected_coord_columns:
        raise ValueError(
            f'Expected {expected_coord_columns} joint coordinate columns, found {len(coord_columns)}.'
        )

    renamed_coord_columns = []
    for joint_idx in range(23):
        renamed_coord_columns.extend([f'X.{joint_idx}', f'Y.{joint_idx}', f'Z.{joint_idx}'])

    rename_map = dict(zip(coord_columns, renamed_coord_columns))
    return data.rename(columns=rename_map)


# Process each xlsx directly without creating temporary csv files
for xlsx_file in xlsx_files:
    xlsx_path = os.path.join(data_path, xlsx_file)

    try:
        xls = pd.ExcelFile(xlsx_path)
        if 'Segment Position' not in xls.sheet_names:
            print(f"Segment Position tab not found in {xlsx_file}. Skipping.")
            continue

        data = pd.read_excel(xls, sheet_name='Segment Position')
        data = rename_awinda_columns(data)
    except Exception as e:
        print(f'Error processing {xlsx_file}: {e}')
        continue

    output_file_name = os.path.splitext(xlsx_file)[0] + '.csv'
    output_file_path = os.path.join(output_path, output_file_name)
    data.to_csv(output_file_path, index=False, sep=',', decimal='.')

    print(f'Renamed columns for {output_file_name}:')
    print(data.head())


Renamed columns for Awinda_001CcSs_3_shadow_test.csv:
   Frame       X.0       Y.0       Z.0       X.1       Y.1       Z.1  \
0      0 -0.584980  1.270854  0.972857 -0.621920  1.226533  1.076873   
1      1 -0.591969  1.279684  0.973840 -0.628194  1.234799  1.077865   
2      2 -0.596741  1.289396  0.972536 -0.632115  1.243965  1.076614   
3      3 -0.600867  1.299259  0.969743 -0.635335  1.253375  1.073925   
4      4 -0.606391  1.307390  0.967172 -0.640211  1.261295  1.071476   

        X.2       Y.2       Z.2  ...      Z.19      X.20      Y.20      Z.20  \
0 -0.605079  1.236162  1.153099  ...  0.982509 -0.661618  1.436022  0.596862   
1 -0.609910  1.243781  1.153859  ...  0.985637 -0.661112  1.459633  0.602973   
2 -0.613035  1.252316  1.152497  ...  0.985933 -0.658354  1.482452  0.606851   
3 -0.615687  1.261206  1.149717  ...  0.984431 -0.654935  1.504334  0.609208   
4 -0.619867  1.269056  1.147080  ...  0.982777 -0.654758  1.521092  0.610357   

       X.21      Y.21      Z.21 

Renamed columns for Awinda_001CcSs_3_shadow_training.csv:
   Frame       X.0       Y.0       Z.0       X.1       Y.1       Z.1  \
0      0 -0.859501  1.636498  0.945990 -0.872375  1.577510  1.048472   
1      1 -0.859623  1.637047  0.945952 -0.872515  1.578004  1.048402   
2      2 -0.859714  1.637611  0.945911 -0.872641  1.578496  1.048314   
3      3 -0.859763  1.638194  0.945864 -0.872747  1.578986  1.048206   
4      4 -0.859781  1.638792  0.945813 -0.872838  1.579473  1.048081   

        X.2       Y.2       Z.2  ...      Z.19      X.20      Y.20      Z.20  \
0 -0.869700  1.595716  1.124967  ...  0.948759 -0.978887  1.640650  0.554197   
1 -0.869867  1.596179  1.124905  ...  0.948737 -0.979010  1.640913  0.554193   
2 -0.870021  1.596624  1.124829  ...  0.948708 -0.979136  1.641144  0.554188   
3 -0.870156  1.597050  1.124737  ...  0.948672 -0.979262  1.641331  0.554184   
4 -0.870279  1.597459  1.124632  ...  0.948630 -0.979392  1.641486  0.554181   

       X.21      Y.21      Z

In [5]:
# Synchronize insole and skeleton files on a shared 60 Hz time grid.
# This avoids nearest-neighbor ambiguity when one device records longer duration.

output_path = CLEAN_DATA_DIR
SAMPLING_HZ = 60.0
GRID_DT = 1.0 / SAMPLING_HZ


def extract_tag(filename, prefix):
    name = os.path.basename(filename)
    stem = os.path.splitext(name)[0]
    if stem.startswith(prefix + '_'):
        return stem.split('_', 1)[1]
    return None


def get_insole_time_series(df):
    if '# time' in df.columns:
        return pd.to_numeric(df['# time'], errors='coerce')
    if 'time' in df.columns:
        return pd.to_numeric(df['time'], errors='coerce')
    raise KeyError("Insole file is missing '# time'/'time' column")


def get_skeleton_time_series(df):
    if '# time' in df.columns:
        return pd.to_numeric(df['# time'], errors='coerce')
    if 'time' in df.columns:
        return pd.to_numeric(df['time'], errors='coerce')
    if 'Timestamp' in df.columns:
        ts = pd.to_datetime(df['Timestamp'], errors='coerce')
        return (ts - ts.iloc[0]).dt.total_seconds()
    if 'Frame' in df.columns:
        return pd.to_numeric(df['Frame'], errors='coerce') / SAMPLING_HZ
    if 'frame' in df.columns:
        return pd.to_numeric(df['frame'], errors='coerce') / SAMPLING_HZ
    # Fallback: assume already sampled at 60 Hz.
    return pd.Series(np.arange(len(df), dtype=np.float64) * GRID_DT)


def prepare_time_and_numeric(df, t_series):
    out = df.copy()
    out['_t'] = pd.to_numeric(t_series, errors='coerce')
    out = out.dropna(subset=['_t']).sort_values('_t')
    out = out[~out['_t'].duplicated(keep='first')]

    value_cols = [c for c in out.columns if c != '_t']
    for c in value_cols:
        out[c] = pd.to_numeric(out[c], errors='coerce')

    return out, value_cols


def interpolate_on_grid(df, value_cols, grid_t):
    x = df['_t'].to_numpy(dtype=np.float64)
    aligned = pd.DataFrame({'# time': grid_t})
    for c in value_cols:
        y = df[c].to_numpy(dtype=np.float64)
        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() < 2:
            aligned[c] = np.nan
            continue
        aligned[c] = np.interp(grid_t, x[valid], y[valid])

    # Fill any all-NaN columns safely.
    aligned = aligned.ffill().bfill()
    return aligned


insole_files = [f for f in os.listdir(output_path) if f.startswith('Soles_')]
skeleton_files = [f for f in os.listdir(output_path) if f.startswith('Awinda_') and f.endswith('.csv')]

insole_map = {}
for f in insole_files:
    tag = extract_tag(f, 'Soles')
    if tag is not None:
        insole_map[tag] = os.path.join(output_path, f)

skeleton_map = {}
for f in skeleton_files:
    tag = extract_tag(f, 'Awinda')
    if tag is not None:
        skeleton_map[tag] = os.path.join(output_path, f)

common_tags = sorted(set(insole_map.keys()).intersection(set(skeleton_map.keys())))
print(f'Found {len(common_tags)} matched Soles/Awinda tags for synchronization.')

for tag in common_tags:
    insole_path = insole_map[tag]
    skeleton_path = skeleton_map[tag]

    try:
        insole_df = pd.read_csv(insole_path, sep=None, engine='python')
        skeleton_df = pd.read_csv(skeleton_path)

        insole_t = get_insole_time_series(insole_df)
        skeleton_t = get_skeleton_time_series(skeleton_df)

        insole_prep, insole_cols = prepare_time_and_numeric(insole_df, insole_t)
        skeleton_prep, skeleton_cols = prepare_time_and_numeric(skeleton_df, skeleton_t)

        # Remove Frame/frame columns from interpolation; regenerate them cleanly after alignment.
        insole_cols = [c for c in insole_cols if c.lower() != 'frame']
        skeleton_cols = [c for c in skeleton_cols if c.lower() != 'frame']

        if insole_prep.empty or skeleton_prep.empty:
            print(f'[skip] {tag}: empty valid time series after parsing.')
            continue

        # Rebase both streams to their own starts, then align on overlap interval.
        insole_start = float(insole_prep['_t'].iloc[0])
        skel_start = float(skeleton_prep['_t'].iloc[0])
        insole_prep['_t'] = insole_prep['_t'] - insole_start
        skeleton_prep['_t'] = skeleton_prep['_t'] - skel_start

        overlap_start = max(float(insole_prep['_t'].iloc[0]), float(skeleton_prep['_t'].iloc[0]))
        overlap_end = min(float(insole_prep['_t'].iloc[-1]), float(skeleton_prep['_t'].iloc[-1]))

        if overlap_end <= overlap_start:
            print(f'[skip] {tag}: no temporal overlap after rebasing.')
            continue

        n_frames = int(np.floor((overlap_end - overlap_start) * SAMPLING_HZ)) + 1
        if n_frames < 2:
            print(f'[skip] {tag}: overlap too short ({overlap_end - overlap_start:.6f}s).')
            continue

        grid_t = overlap_start + np.arange(n_frames, dtype=np.float64) * GRID_DT

        insole_aligned = interpolate_on_grid(insole_prep, insole_cols, grid_t)
        skeleton_aligned = interpolate_on_grid(skeleton_prep, skeleton_cols, grid_t)

        # Keep a clean frame index for downstream code.
        if 'Frame' in insole_aligned.columns:
            insole_aligned['Frame'] = np.arange(len(insole_aligned), dtype=np.int64)
        else:
            insole_aligned.insert(0, 'Frame', np.arange(len(insole_aligned), dtype=np.int64))

        if 'Frame' in skeleton_aligned.columns:
            skeleton_aligned['Frame'] = np.arange(len(skeleton_aligned), dtype=np.int64)
        else:
            skeleton_aligned.insert(0, 'Frame', np.arange(len(skeleton_aligned), dtype=np.int64))

        insole_aligned.to_csv(insole_path, index=False)
        skeleton_aligned.to_csv(skeleton_path, index=False)

        print(
            f"[aligned-grid] {tag}: insole {len(insole_df)} -> {len(insole_aligned)}, "
            f"skeleton {len(skeleton_df)} -> {len(skeleton_aligned)}, "
            f"overlap={overlap_end - overlap_start:.3f}s"
        )

    except Exception as e:
        print(f'[error] {tag}: {e}')

Found 2 matched Soles/Awinda tags for synchronization.
[aligned-grid] 001CcSs_3_shadow_test: insole 1002 -> 501, skeleton 501 -> 501, overlap=8.333s


[aligned-grid] 001CcSs_3_shadow_training: insole 5000 -> 2501, skeleton 2501 -> 2501, overlap=41.667s


## Translationally normalize joints' positions to pelvis position

In [6]:
# Position hold processing
# Use pelvis joint (X.0, Y.0, Z.0) to normalize/align skeleton coordinate columns

def normalize_skeleton_data(df):
    pelvis_cols = ['X.0', 'Y.0', 'Z.0']
    if not all(col in df.columns for col in pelvis_cols):
        raise ValueError(f'Missing pelvis columns: {pelvis_cols}')

    pelvis_data = df[pelvis_cols]
    normalized_df = df.copy()

    for col in df.columns:
        if col.startswith(('X.', 'Y.', 'Z.')):
            axis = col.split('.')[0]
            normalized_df[col] = df[col] - pelvis_data[f'{axis}.0']

    return normalized_df


# Get all csv files in the output folder
csv_files = [f for f in os.listdir(output_path) if f.endswith('.csv')]

# Process each CSV file in the clean_data directory and overwrite in place
for csv_file in csv_files:
    file_path = os.path.join(output_path, csv_file)
    try:
        data = pd.read_csv(file_path)
        normalized_data = normalize_skeleton_data(data)
    except Exception as e:
        print(f'Error normalizing {csv_file}: {e}')
        continue

    # Safe overwrite: write to temp file, then atomically replace original
    tmp_file_path = file_path + '.tmp'
    normalized_data.to_csv(tmp_file_path, index=False, sep=',', decimal='.')
    os.replace(tmp_file_path, file_path)

    print(f'Overwrote normalized skeleton data in {file_path}.')


Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_3_shadow_test.csv.


Overwrote normalized skeleton data in c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_001CcSs_3_shadow_training.csv.


## SoleFormer target prep (Awinda raw tabs)
Use this section to extract all tabs from raw Awinda XLSX files and keep only the tabs required for modeling.
It writes to data/raw_data/Awinda/extracted_tabs and then builds merged targets from the selected tab CSVs.

In [7]:
from pathlib import Path
import re
import pandas as pd


def _safe_sheet_name(sheet_name: str) -> str:
    # Keep filenames stable across locales and special characters.
    cleaned = re.sub(r"[^0-9A-Za-z]+", "_", str(sheet_name).strip())
    return cleaned.strip("_") or "Sheet"


def _split_awinda_stem(stem: str):
    # Example stem: Awinda_001CcSs_2_tech_paos
    # Returns: (prefix='Awinda', tag='001CcSs_2_tech_paos')
    if "_" not in stem:
        raise ValueError(f"Unexpected Awinda filename format: {stem}")
    prefix, tag = stem.split("_", 1)
    return prefix, tag


def extract_awinda_tabs_from_xlsx(
    awinda_xlsx_dir,
    output_root_dir,
    required_sheet_keywords=None,
    exclude_sheet_keywords=None,
    drop_frame_columns=False,
):
    """
    Extract tabs from each Awinda XLSX into CSV.

    Output structure:
      - <output_root>/all_tabs/*.csv        (all sheets from all files)
      - <output_root>/required_tabs/*.csv   (only selected sheets)

    Matching is based on sheet names (keywords), not tab indices,
    so it remains robust if tab numbers are shifted (e.g., missing tab2).
    """
    awinda_xlsx_dir = Path(awinda_xlsx_dir)
    output_root_dir = Path(output_root_dir)
    all_tabs_dir = output_root_dir / "all_tabs"
    required_tabs_dir = output_root_dir / "required_tabs"
    all_tabs_dir.mkdir(parents=True, exist_ok=True)
    required_tabs_dir.mkdir(parents=True, exist_ok=True)

    if required_sheet_keywords is None:
        required_sheet_keywords = [
            "Segment Position",
            "Joint Angles ZXY",
        ]

    if exclude_sheet_keywords is None:
        exclude_sheet_keywords = [
            "Ergonomic Joint Angles ZXY",
        ]

    required_lower = [k.lower() for k in required_sheet_keywords]
    exclude_lower = [k.lower() for k in exclude_sheet_keywords]

    xlsx_files = sorted(awinda_xlsx_dir.glob("Awinda_*.xlsx"))
    if not xlsx_files:
        raise FileNotFoundError(f"No Awinda XLSX files found in {awinda_xlsx_dir}")

    exported_all = 0
    exported_required = 0

    for xlsx_path in xlsx_files:
        stem = xlsx_path.stem
        _, tag = _split_awinda_stem(stem)

        xl = pd.ExcelFile(xlsx_path, engine="openpyxl")
        for sheet_index, sheet_name in enumerate(xl.sheet_names, start=1):
            df = pd.read_excel(xlsx_path, sheet_name=sheet_name, engine="openpyxl")
            df.columns = [str(c).strip() for c in df.columns]

            if drop_frame_columns:
                df = df.drop(columns=["Frame", "frame"], errors="ignore")

            safe_sheet = _safe_sheet_name(sheet_name)
            out_name = f"Awinda_{tag}_tab{sheet_index}_{safe_sheet}.csv"
            out_all = all_tabs_dir / out_name
            df.to_csv(out_all, index=False)
            exported_all += 1

            sheet_lower = sheet_name.lower()
            include_required = any(keyword in sheet_lower for keyword in required_lower)
            exclude_match = any(keyword in sheet_lower for keyword in exclude_lower)
            if include_required and not exclude_match:
                out_req = required_tabs_dir / out_name
                df.to_csv(out_req, index=False)
                exported_required += 1

    print(f"Extracted all tabs: {exported_all} CSV files into {all_tabs_dir}")
    print(f"Extracted required tabs: {exported_required} CSV files into {required_tabs_dir}")
    print(f"Required sheet keywords: {required_sheet_keywords}")
    print(f"Excluded sheet keywords: {exclude_sheet_keywords}")

    return all_tabs_dir, required_tabs_dir


def _read_awinda_tab_csv(path):
    # Try semicolon first, fallback to comma.
    df = pd.read_csv(path, sep=";", engine="python")
    if df.shape[1] == 1:
        df = pd.read_csv(path, sep=",", engine="python")

    df.columns = [c.strip() for c in df.columns]
    df = df.drop(columns=["Frame", "frame"], errors="ignore")
    return df


def _find_single_tab_csv(tab_dir, tag, pattern):
    matches = sorted(tab_dir.glob(f"Awinda_{tag}_{pattern}.csv"))
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one match for tag={tag}, pattern={pattern}, found {len(matches)}"
        )
    return matches[0]


def build_soleformer_targets_from_tabs(
    insole_dir,
    awinda_required_tab_dir,
    out_dir,
    include_positions=True,
    include_joint_angles=True,
    position_pattern="*Segment_Position*",
    angle_pattern="*Joint_Angles_ZXY*",
):
    """Merge selected tab CSVs into model target files aligned by insole tags."""
    insole_dir = Path(insole_dir)
    awinda_required_tab_dir = Path(awinda_required_tab_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    insole_files = sorted(insole_dir.glob("Soles_*.txt"))
    if not insole_files:
        raise FileNotFoundError(f"No insole files found in {insole_dir}")

    for insole_path in insole_files:
        tag = insole_path.stem.split("_", 1)[1]
        parts = []

        if include_positions:
            pos_path = _find_single_tab_csv(awinda_required_tab_dir, tag, position_pattern)
            pos_df = _read_awinda_tab_csv(pos_path).add_prefix("pos::")
            parts.append(pos_df)

        if include_joint_angles:
            ang_path = _find_single_tab_csv(awinda_required_tab_dir, tag, angle_pattern)
            ang_df = _read_awinda_tab_csv(ang_path).add_prefix("ang::")
            parts.append(ang_df)

        if not parts:
            raise ValueError("At least one target source must be enabled.")

        merged = pd.concat(parts, axis=1)
        out_path = out_dir / f"AwindaTarget_{tag}.csv"
        merged.to_csv(out_path, index=False)
        print(f"Saved {out_path} with shape {merged.shape}")

In [8]:
# Example usage
raw_awinda_dir = Path(PROJECT_ROOT) / "data" / "raw_data" / "Awinda"
extracted_root = raw_awinda_dir / "extracted_tabs"

# 1) Extract all tabs from each XLSX, and also copy only tabs needed for model targets
_, required_tab_dir = extract_awinda_tabs_from_xlsx(
    awinda_xlsx_dir=raw_awinda_dir,
    output_root_dir=extracted_root,
    required_sheet_keywords=[
        "Segment Position",
        "Joint Angles ZXY",
    ],
    exclude_sheet_keywords=[
        "Ergonomic Joint Angles ZXY",
    ],
    drop_frame_columns=False,
)

# 1b) Synchronize extracted tabs to match insole/skeleton time grid
# Import the synchronization module
import sys
sys.path.insert(0, str(Path(PROJECT_ROOT) / "sources" / "usefull_tools"))
from awinda_tab_sync import synchronize_awinda_tabs_to_insole_grid

synchronized_tab_dir = Path(PROJECT_ROOT) / "data" / "raw_data" / "Awinda" / "synchronized_tabs"
required_tab_dir = synchronize_awinda_tabs_to_insole_grid(
    clean_data_dir=Path(PROJECT_ROOT) / "data" / "clean_data",
    insole_dir=Path(PROJECT_ROOT) / "data" / "clean_data",
    awinda_required_tab_dir=required_tab_dir,
    output_dir=synchronized_tab_dir,
    sampling_hz=60.0,
)

# 2) Build merged target CSVs (positions + angles) from synchronized tabs
build_soleformer_targets_from_tabs(
    insole_dir=Path(PROJECT_ROOT) / "data" / "clean_data",
    awinda_required_tab_dir=required_tab_dir,
    out_dir=Path(PROJECT_ROOT) / "data" / "clean_data" / "Awinda_targets_soleformer",
    include_positions=True,
    include_joint_angles=True,
    position_pattern="*Segment_Position*",
    angle_pattern="*Joint_Angles_ZXY*",
)


Extracted all tabs: 36 CSV files into c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\raw_data\Awinda\extracted_tabs\all_tabs
Extracted required tabs: 4 CSV files into c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\raw_data\Awinda\extracted_tabs\required_tabs
Required sheet keywords: ['Segment Position', 'Joint Angles ZXY']
Excluded sheet keywords: ['Ergonomic Joint Angles ZXY']
[sync] 001CcSs_3_shadow_test: position 501 -> 501, angles 501 -> 501


[sync] 001CcSs_3_shadow_training: position 2501 -> 2501, angles 2501 -> 2501

Synchronized 2 Awinda tab pairs to insole row count.
Saved c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_targets_soleformer\AwindaTarget_001CcSs_3_shadow_test.csv with shape (501, 135)


Saved c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\clean_data\Awinda_targets_soleformer\AwindaTarget_001CcSs_3_shadow_training.csv with shape (2501, 135)


## Runtime split for sources/main.py (train/test) and validation
This section bridges the gap between preprocessed clean files and the folder structure consumed by the principal pipeline (`sources/main.py`).

What it does:
- Pairs clean `Awinda_*.csv` and `Soles_*.txt` by tag
- Routes `_training` tags into `data/training_data`
- Routes `_test` tags into `data/test_data`
- Keeps one active test pair at top-level and moves additional test pairs into `Other testing files`
- Validates pair availability and SoleFormer target availability (`AwindaTarget_<tag>.csv`)

In [9]:
import shutil
from pathlib import Path


def _extract_tag_from_name(path_obj, prefix):
    stem = Path(path_obj).stem
    if stem.startswith(prefix + '_'):
        return stem.split('_', 1)[1]
    return None


def _split_from_tag(tag):
    tag_lower = str(tag).lower()
    if '_training' in tag_lower:
        return 'training'
    if '_test' in tag_lower:
        return 'test'
    return 'training'


def _ensure_dir(path_obj):
    Path(path_obj).mkdir(parents=True, exist_ok=True)


def _clear_top_level_files(folder, pattern):
    folder = Path(folder)
    for p in folder.glob(pattern):
        if p.is_file():
            p.unlink()


def route_clean_data_to_runtime_structure(project_root):
    project_root = Path(project_root)
    clean_dir = project_root / 'data' / 'clean_data'

    train_insole_dir = project_root / 'data' / 'training_data' / 'Insole'
    train_skel_dir = project_root / 'data' / 'training_data' / 'skeleton'
    test_insole_dir = project_root / 'data' / 'test_data' / 'Insole'
    test_skel_dir = project_root / 'data' / 'test_data' / 'skeleton'
    test_insole_other_dir = test_insole_dir / 'Other testing files'
    test_skel_other_dir = test_skel_dir / 'Other testing files'

    for d in [
        train_insole_dir,
        train_skel_dir,
        test_insole_dir,
        test_skel_dir,
        test_insole_other_dir,
        test_skel_other_dir,
    ]:
        _ensure_dir(d)

    insole_files = sorted([p for p in clean_dir.glob('Soles_*.txt') if p.is_file()])
    skeleton_files = sorted([p for p in clean_dir.glob('Awinda_*.csv') if p.is_file()])

    insole_map = {}
    for p in insole_files:
        tag = _extract_tag_from_name(p, 'Soles')
        if tag:
            insole_map[tag] = p

    skeleton_map = {}
    for p in skeleton_files:
        tag = _extract_tag_from_name(p, 'Awinda')
        if tag:
            skeleton_map[tag] = p

    common_tags = sorted(set(insole_map.keys()) & set(skeleton_map.keys()))
    if not common_tags:
        raise ValueError(
            f'No paired clean files found in {clean_dir}. '
            'Expected matching Soles_<tag>.txt and Awinda_<tag>.csv.'
        )

    # Clean runtime top-level destinations before copying fresh active files.
    _clear_top_level_files(train_insole_dir, 'Soles_*.txt')
    _clear_top_level_files(train_skel_dir, 'Awinda_*.csv')
    _clear_top_level_files(test_insole_dir, 'Soles_*.txt')
    _clear_top_level_files(test_skel_dir, 'Awinda_*.csv')
    _clear_top_level_files(test_insole_other_dir, 'Soles_*.txt')
    _clear_top_level_files(test_skel_other_dir, 'Awinda_*.csv')

    training_tags = []
    test_tags = []
    for tag in common_tags:
        split = _split_from_tag(tag)
        if split == 'test':
            test_tags.append(tag)
        else:
            training_tags.append(tag)

    # Copy all training pairs to top-level training folders.
    for tag in training_tags:
        shutil.copy2(insole_map[tag], train_insole_dir / insole_map[tag].name)
        shutil.copy2(skeleton_map[tag], train_skel_dir / skeleton_map[tag].name)

    # Keep one active test pair top-level for predict/visual defaults, store extras in Other testing files.
    active_test_tag = None
    if test_tags:
        active_test_tag = test_tags[0]
        shutil.copy2(insole_map[active_test_tag], test_insole_dir / insole_map[active_test_tag].name)
        shutil.copy2(skeleton_map[active_test_tag], test_skel_dir / skeleton_map[active_test_tag].name)

        for tag in test_tags[1:]:
            shutil.copy2(insole_map[tag], test_insole_other_dir / insole_map[tag].name)
            shutil.copy2(skeleton_map[tag], test_skel_other_dir / skeleton_map[tag].name)

    print('Runtime data routing completed.')
    print(f'  Training pairs copied: {len(training_tags)} -> {train_insole_dir} and {train_skel_dir}')
    print(f'  Test pairs found: {len(test_tags)}')
    if active_test_tag is not None:
        print(f'  Active test tag at top-level: {active_test_tag}')
    else:
        print('  No test-tagged file found. Predict/visual default lookup may fail unless you pass CLI file paths.')

    return {
        'training_tags': training_tags,
        'test_tags': test_tags,
        'active_test_tag': active_test_tag,
    }


def validate_runtime_readiness(project_root, routing_summary):
    project_root = Path(project_root)

    train_insole_dir = project_root / 'data' / 'training_data' / 'Insole'
    train_skel_dir = project_root / 'data' / 'training_data' / 'skeleton'
    test_insole_dir = project_root / 'data' / 'test_data' / 'Insole'
    test_skel_dir = project_root / 'data' / 'test_data' / 'skeleton'
    soleformer_target_dir = project_root / 'data' / 'clean_data' / 'Awinda_targets_soleformer'

    train_insole_top = sorted(train_insole_dir.glob('Soles_*.txt'))
    train_skel_top = sorted(train_skel_dir.glob('Awinda_*.csv'))
    test_insole_top = sorted(test_insole_dir.glob('Soles_*.txt'))
    test_skel_top = sorted(test_skel_dir.glob('Awinda_*.csv'))

    if not train_insole_top or not train_skel_top:
        raise ValueError('Training top-level folders are missing files. Run preprocessing cells and check tag parsing.')

    if not test_insole_top or not test_skel_top:
        print('Warning: Test top-level folders are missing files. Add *_test files or pass explicit CLI paths for predict/visual.')

    # SoleFormer needs target files for training tags.
    missing_targets = []
    for tag in routing_summary['training_tags']:
        expected = soleformer_target_dir / f'AwindaTarget_{tag}.csv'
        if not expected.is_file():
            missing_targets.append(expected.name)

    print('Readiness checks:')
    print(f'  train top-level insole files: {len(train_insole_top)}')
    print(f'  train top-level skeleton files: {len(train_skel_top)}')
    print(f'  test top-level insole files: {len(test_insole_top)}')
    print(f'  test top-level skeleton files: {len(test_skel_top)}')

    if missing_targets:
        print('Warning: missing SoleFormer merged targets for some training tags:')
        for name in missing_targets:
            print(f'  - {name}')
    else:
        print('  SoleFormer targets: OK for all training tags.')


# Execute routing + validation.
routing = route_clean_data_to_runtime_structure(PROJECT_ROOT)
validate_runtime_readiness(PROJECT_ROOT, routing)

Runtime data routing completed.
  Training pairs copied: 1 -> c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\training_data\Insole and c:\Users\joels\Desktop\IEAP-Projects\Python-R-Git\Project\singer.joel\data\training_data\skeleton
  Test pairs found: 1
  Active test tag at top-level: 001CcSs_3_shadow_test
Readiness checks:
  train top-level insole files: 1
  train top-level skeleton files: 1
  test top-level insole files: 1
  test top-level skeleton files: 1
  SoleFormer targets: OK for all training tags.
